# Phase 6: Feature Steering

**Interpretability of Portuguese Language Models via Sparse Autoencoders**

---

**Goal:** Establish causal relationships between features and model behavior, and demonstrate control of textual register.

**What this notebook does:**
1. Loads model, SAE, and selected features (from Phases 3-5)
2. Implements the **steering mechanism**: feature injection/suppression during the forward pass
3. **Ablation experiments**: suppresses PT-specific features (crase, clitics, gender) and measures the effect on generation
4. **Register amplification experiments**: amplifies textual-register features (legal, scientific, colloquial, journalistic) and evaluates tone control
5. **Quantitative evaluation**: measures changes in token distribution and coherence
6. Saves results and generates comparative visualizations (before/after)

**Prerequisites:**
- GPU with ≥16GB VRAM (Colab T4)
- Checkpoints from the previous phases (`stats_*.pt`, `phase4_probes_results.pt`)
- Estimated runtime: ~30-45 min on T4

In [ ]:
!pip install sae-lens transformer-lens -q
print()
print("Installation complete. Restart the runtime before continuing:")
print("  Runtime → Restart session (ou Ctrl+M .)")
print("  Then skip this cell and run from the next one.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 10
from tqdm.auto import tqdm
import time
import os
import json
import copy

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name()
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print('Usando Apple Silicon (MPS)')
else:
    device = 'cpu'
    print('No GPU detected.')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from sae_lens import SAE, HookedSAETransformer

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"

print("Loading Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b",
    device=device,
    dtype=torch.float16,
)
print(f"Model: {model.cfg.model_name} | Layers: {model.cfg.n_layers} | d_model: {model.cfg.d_model}")

print()
print(f"Loading SAE: {SAE_ID}...")
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)

HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
print(f"SAE: {sae.cfg.d_sae} features | d_in: {sae.cfg.d_in} | hook: {HOOK_NAME}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SAVE_DIR = "./checkpoints/"  # adjust to your preferred path (e.g., ./checkpoints/ for Colab)

FEATURES_PT = {
    "crase": 4584,
    "enclise": 2817,
    "proclise": 6215,
    "genero_feminino": 1215,
    "infinitivo_pessoal": 10349,
    "contracoes": 2294,
}

FEATURES_REGISTRO = {
    "coloquial": 1082,
    "cientifico": 5880,
    "juridico_1": 2294,
    "juridico_2": 12269,
    "jornalistico_1": 16057,
    "jornalistico_2": 10478,
    "coloquial_2": 15135,
}

print("PT-specific features for ablation:")
for name, fid in FEATURES_PT.items():
    print(f"  {name}: Feature {fid}")

print()
print("Register features for amplification:")
for name, fid in FEATURES_REGISTRO.items():
    print(f"  {name}: Feature {fid}")

## 1. Feature Steering Mechanism

Steering works by intercepting the forward pass at the SAE layer (layer 13, residual stream):

1. The model processes tokens normally up to layer 13
2. We intercept the residual-stream activations
3. We pass them through the SAE encoder to obtain feature activations
4. We **modify** the target feature activations (zero for ablation, multiply for amplification)
5. We pass them through the SAE decoder to reconstruct the modified residual stream
6. The model continues the forward pass with the altered activations

**Important:** Steering is applied token by token during autoregressive generation.

In [ ]:
def generate_with_steering(model, sae, tokenizer, prompt, feature_ids,
                          multipliers, max_new_tokens=100, temperature=0.7,
                          hook_name=HOOK_NAME):
    """
    Generate text with feature steering applied at the SAE layer.

    Args:
        feature_ids: list of feature indices to steer
        multipliers: list of multipliers (0.0 = ablate, >1.0 = amplify, <0 = suppress)
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    feature_ids_t = torch.tensor(feature_ids, device=device)
    multipliers_t = torch.tensor(multipliers, dtype=torch.float16, device=device)

    def steering_hook(value, hook):
        with torch.no_grad():
            sae_input = value
            sae_acts = sae.encode(sae_input)
            modified_acts = sae_acts.clone()
            for fid, mult in zip(feature_ids_t, multipliers_t):
                modified_acts[:, :, fid] = modified_acts[:, :, fid] * mult
            reconstructed = sae.decode(modified_acts)
            error = sae_input - sae.decode(sae_acts)
            return reconstructed + error

    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model.run_with_hooks(
                generated,
                fwd_hooks=[(hook_name, steering_hook)],
            )
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)

            if next_token.item() == tokenizer.eos_token_id:
                break

            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)

    result = tokenizer.decode(generated[0], skip_special_tokens=True)
    return result


def generate_baseline(model, tokenizer, prompt, max_new_tokens=100,
                      temperature=0.7):
    """Generate text without any steering (baseline)."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(generated)
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)

            if next_token.item() == tokenizer.eos_token_id:
                break

            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)

    result = tokenizer.decode(generated[0], skip_special_tokens=True)
    return result


tokenizer = model.tokenizer
print("Mecanismo de steering implementado.")
print(f"Tokenizer: {tokenizer.__class__.__name__} | Vocab: {tokenizer.vocab_size}")

## 2. Ablation Experiments

We suppress PT-specific features (multiplier = 0.0) and observe the effect on generation.

**Hypotheses:**
- Suppressing the crase feature (4584) → reduces correct crase usage
- Suppressing the feminine gender feature (1215) → affects gender agreement
- Suppressing the enclisis feature (2817) → shifts clitic position

In [ ]:
print("=" * 70)
print("EXPERIMENT 1: CRASE FEATURE ABLATION (Feature 4584)")
print("=" * 70)

prompts_crase = [
    "Eu vou à",
    "A proposta foi encaminhada à",
    "O aluno foi à escola e depois voltou à",
    "Refiro-me à questão levantada pela",
]

torch.manual_seed(42)
results_crase = []

for prompt in prompts_crase:
    print(f"\nPrompt: \"{prompt}\"")
    print("-" * 50)

    baseline = generate_baseline(model, tokenizer, prompt,
                                 max_new_tokens=50, temperature=0.3)
    print(f"  Baseline:  {baseline}")

    steered = generate_with_steering(model, sae, tokenizer, prompt,
                                     feature_ids=[FEATURES_PT["crase"]],
                                     multipliers=[0.0],
                                     max_new_tokens=50, temperature=0.3)
    print(f"  Ablation:  {steered}")

    results_crase.append({
        "prompt": prompt,
        "baseline": baseline,
        "steered": steered,
        "feature": 4584,
        "operation": "ablation",
    })

print()
print("Crase ablation complete.")

In [ ]:
print("=" * 70)
print("EXPERIMENT 2: FEMININE GENDER FEATURE ABLATION (Feature 1215)")
print("=" * 70)

prompts_genero = [
    "A menina bonita foi à",
    "A professora explicou que a aluna era muito",
    "Ela é uma pessoa muito",
    "A diretora da empresa apresentou a nova",
]

torch.manual_seed(42)
results_genero = []

for prompt in prompts_genero:
    print(f"\nPrompt: \"{prompt}\"")
    print("-" * 50)

    baseline = generate_baseline(model, tokenizer, prompt,
                                 max_new_tokens=50, temperature=0.3)
    print(f"  Baseline:  {baseline}")

    steered = generate_with_steering(model, sae, tokenizer, prompt,
                                     feature_ids=[FEATURES_PT["genero_feminino"]],
                                     multipliers=[0.0],
                                     max_new_tokens=50, temperature=0.3)
    print(f"  Ablation:  {steered}")

    results_genero.append({
        "prompt": prompt,
        "baseline": baseline,
        "steered": steered,
        "feature": 1215,
        "operation": "ablation",
    })

print()
print("Feminine gender ablation complete.")

In [ ]:
print("=" * 70)
print("EXPERIMENT 3: CLITIC FEATURES ABLATION (2817 + 6215)")
print("=" * 70)

prompts_cliticos = [
    "Diga-me o que",
    "Ele apresentou-se como",
    "Faça-me um favor e",
    "Entregou-lhe o documento e",
]

torch.manual_seed(42)
results_cliticos = []

for prompt in prompts_cliticos:
    print(f"\nPrompt: \"{prompt}\"")
    print("-" * 50)

    baseline = generate_baseline(model, tokenizer, prompt,
                                 max_new_tokens=50, temperature=0.3)
    print(f"  Baseline:  {baseline}")

    steered = generate_with_steering(model, sae, tokenizer, prompt,
                                     feature_ids=[FEATURES_PT["enclise"],
                                                  FEATURES_PT["proclise"]],
                                     multipliers=[0.0, 0.0],
                                     max_new_tokens=50, temperature=0.3)
    print(f"  Ablation:  {steered}")

    results_cliticos.append({
        "prompt": prompt,
        "baseline": baseline,
        "steered": steered,
        "feature": [2817, 6215],
        "operation": "ablation",
    })

print()
print("Clitics ablation complete.")

## 3. Register Amplification Experiments

We amplify textual-register features (multiplier > 1.0) to control the tone of generation.

**Hypotheses:**
- Amplifying the legal feature (2294, 12269) → text with a more formal/legal tone
- Amplifying the scientific feature (5880) → text with a more academic tone
- Amplifying the colloquial feature (1082) → text with a more informal tone
- Amplifying the journalistic feature (16057, 10478) → text with a more news-like tone

**Strategy:** We use the same neutral prompt and vary only the amplified register.

In [ ]:
print("=" * 70)
print("EXPERIMENT 4: REGISTER CONTROL — SAME PROMPT, DIFFERENT REGISTERS")
print("=" * 70)

REGISTER_MULT = 8.0

prompts_register = [
    "O governo anunciou novas medidas para",
    "A pesquisa sobre inteligência artificial mostra que",
    "O problema da educação no Brasil é que",
    "Os dados indicam que a economia brasileira",
]

register_configs = {
    "baseline": {"ids": [], "mults": []},
    "juridico": {
        "ids": [FEATURES_REGISTRO["juridico_1"], FEATURES_REGISTRO["juridico_2"]],
        "mults": [REGISTER_MULT, REGISTER_MULT],
    },
    "cientifico": {
        "ids": [FEATURES_REGISTRO["cientifico"]],
        "mults": [REGISTER_MULT],
    },
    "coloquial": {
        "ids": [FEATURES_REGISTRO["coloquial"], FEATURES_REGISTRO["coloquial_2"]],
        "mults": [REGISTER_MULT, REGISTER_MULT],
    },
    "jornalistico": {
        "ids": [FEATURES_REGISTRO["jornalistico_1"], FEATURES_REGISTRO["jornalistico_2"]],
        "mults": [REGISTER_MULT, REGISTER_MULT],
    },
}

torch.manual_seed(42)
results_registro = []

for prompt in prompts_register:
    print(f"\nPrompt: \"{prompt}\"")
    print("=" * 60)

    for reg_name, reg_cfg in register_configs.items():
        if reg_name == "baseline":
            text = generate_baseline(model, tokenizer, prompt,
                                     max_new_tokens=80, temperature=0.3)
        else:
            text = generate_with_steering(model, sae, tokenizer, prompt,
                                          feature_ids=reg_cfg["ids"],
                                          multipliers=reg_cfg["mults"],
                                          max_new_tokens=80, temperature=0.3)
        label = reg_name.upper()
        print(f"  [{label:>13}] {text}")

        results_registro.append({
            "prompt": prompt,
            "register": reg_name,
            "text": text,
            "features": reg_cfg["ids"],
            "multiplier": REGISTER_MULT if reg_name != "baseline" else 0,
        })

    print()

print("Register control complete.")

## 4. Multiplier Sweep

To understand the ideal steering intensity, we vary the multiplier from 0× to 16× for one register feature and evaluate the progressive effect.

In [ ]:
print("=" * 70)
print("EXPERIMENT 5: MULTIPLIER SWEEP — LEGAL FEATURE (2294)")
print("=" * 70)

sweep_prompt = "O réu foi"
sweep_multipliers = [0.0, 1.0, 2.0, 4.0, 8.0, 12.0, 16.0]

torch.manual_seed(42)
results_sweep = []

for mult in sweep_multipliers:
    if mult == 1.0:
        text = generate_baseline(model, tokenizer, sweep_prompt,
                                 max_new_tokens=80, temperature=0.3)
        label = "1.0x (baseline)"
    elif mult == 0.0:
        text = generate_with_steering(model, sae, tokenizer, sweep_prompt,
                                      feature_ids=[FEATURES_REGISTRO["juridico_1"]],
                                      multipliers=[0.0],
                                      max_new_tokens=80, temperature=0.3)
        label = "0.0x (ablado)"
    else:
        text = generate_with_steering(model, sae, tokenizer, sweep_prompt,
                                      feature_ids=[FEATURES_REGISTRO["juridico_1"]],
                                      multipliers=[mult],
                                      max_new_tokens=80, temperature=0.3)
        label = f"{mult:.0f}x"

    print(f"  [{label:>15}] {text}")
    results_sweep.append({
        "multiplier": mult,
        "text": text,
        "feature": 2294,
    })

print()
print("Multiplier sweep complete.")

## 5. Quantitative Activation Analysis

For each experiment, we measure the SAE activations on the generated text and compare:
- Mean activation of the target features (baseline vs steered)
- Distribution of active features

In [ ]:
def measure_feature_activations(model, sae, tokenizer, text, feature_ids,
                               hook_name=HOOK_NAME):
    """Measure SAE feature activations for a given text."""
    input_ids = tokenizer.encode(text, return_tensors="pt").to(device)
    activations = {}

    def capture_hook(value, hook):
        activations["resid"] = value.detach()
        return value

    with torch.no_grad():
        model.run_with_hooks(input_ids, fwd_hooks=[(hook_name, capture_hook)])

    resid = activations["resid"]
    sae_acts = sae.encode(resid)

    results = {}
    for fid in feature_ids:
        acts = sae_acts[0, :, fid].detach().cpu().numpy()
        results[fid] = {
            "mean": float(np.mean(acts)),
            "max": float(np.max(acts)),
            "nonzero_frac": float(np.mean(acts > 0)),
        }

    total_active = (sae_acts[0] > 0).sum(dim=-1).float()
    results["total_active_mean"] = float(total_active.mean().cpu())
    results["total_active_std"] = float(total_active.std().cpu())

    return results


print("Quantitative analysis: Crase Ablation")
print("-" * 50)

for r in results_crase[:2]:
    print(f"\nPrompt: \"{r['prompt']}\"")

    base_acts = measure_feature_activations(
        model, sae, tokenizer, r["baseline"], [4584])
    steer_acts = measure_feature_activations(
        model, sae, tokenizer, r["steered"], [4584])

    print(f"  Baseline  — Feature 4584: mean={base_acts[4584]['mean']:.3f}, "
          f"max={base_acts[4584]['max']:.3f}, "
          f"active={base_acts[4584]['nonzero_frac']*100:.0f}%")
    print(f"  Ablation  — Feature 4584: mean={steer_acts[4584]['mean']:.3f}, "
          f"max={steer_acts[4584]['max']:.3f}, "
          f"active={steer_acts[4584]['nonzero_frac']*100:.0f}%")
    print(f"  Active features (baseline): {base_acts['total_active_mean']:.0f} ± "
          f"{base_acts['total_active_std']:.0f}")
    print(f"  Active features (ablation):  {steer_acts['total_active_mean']:.0f} ± "
          f"{steer_acts['total_active_std']:.0f}")

print()
print("Quantitative analysis: Register Amplification")
print("-" * 50)

prompt_test = prompts_register[0]
for reg_name in ["baseline", "juridico", "cientifico", "coloquial"]:
    matching = [r for r in results_registro
                if r["prompt"] == prompt_test and r["register"] == reg_name]
    if matching:
        r = matching[0]
        fids = list(FEATURES_REGISTRO.values())
        acts = measure_feature_activations(model, sae, tokenizer, r["text"], fids)
        print(f"\n  [{reg_name.upper():>13}]")
        for fname, fid in FEATURES_REGISTRO.items():
            if fid in acts:
                print(f"    Feature {fid} ({fname}): "
                      f"mean={acts[fid]['mean']:.2f}, max={acts[fid]['max']:.2f}")

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

experiments = [
    ("Crase (F4584)", results_crase, [4584]),
    ("Gênero (F1215)", results_genero, [1215]),
    ("Clíticos (F2817+6215)", results_cliticos, [2817, 6215]),
]

for ax, (title, results, fids) in zip(axes, experiments):
    base_means = []
    steer_means = []

    for r in results:
        b_acts = measure_feature_activations(model, sae, tokenizer, r["baseline"], fids)
        s_acts = measure_feature_activations(model, sae, tokenizer, r["steered"], fids)

        base_val = np.mean([b_acts[f]["mean"] for f in fids])
        steer_val = np.mean([s_acts[f]["mean"] for f in fids])
        base_means.append(base_val)
        steer_means.append(steer_val)

    x = np.arange(len(results))
    width = 0.35
    ax.bar(x - width/2, base_means, width, label="Baseline", color="#2196F3")
    ax.bar(x + width/2, steer_means, width, label="Ablação (0x)", color="#F44336")
    ax.set_title(f"Ablação: {title}")
    ax.set_xlabel("Prompt")
    ax.set_ylabel("Ativação média")
    ax.set_xticks(x)
    ax.set_xticklabels([f"P{i+1}" for i in x])
    ax.legend()

plt.tight_layout()
plt.savefig("fig_phase6_ablation_comparison.png", dpi=150, bbox_inches="tight")
if os.path.exists(SAVE_DIR):
    plt.savefig(SAVE_DIR + "fig_phase6_ablation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: fig_phase6_ablation_comparison.png")

In [ ]:
fig, axes = plt.subplots(1, len(prompts_register), figsize=(5*len(prompts_register), 5))
if len(prompts_register) == 1:
    axes = [axes]

registers_order = ["juridico", "cientifico", "coloquial", "jornalistico"]
colors = {"juridico": "#8B0000", "cientifico": "#1565C0",
          "coloquial": "#FF9800", "jornalistico": "#2E7D32"}

for ax, prompt in zip(axes, prompts_register):
    register_data = {}
    for reg in registers_order:
        matching = [r for r in results_registro
                    if r["prompt"] == prompt and r["register"] == reg]
        if matching:
            fids = list(FEATURES_REGISTRO.values())
            acts = measure_feature_activations(model, sae, tokenizer,
                                               matching[0]["text"], fids)
            register_data[reg] = {fname: acts[fid]["mean"]
                                  for fname, fid in FEATURES_REGISTRO.items()
                                  if fid in acts}

    feat_names = list(FEATURES_REGISTRO.keys())
    x = np.arange(len(feat_names))
    width = 0.2

    for i, reg in enumerate(registers_order):
        if reg in register_data:
            vals = [register_data[reg].get(fn, 0) for fn in feat_names]
            ax.bar(x + i*width, vals, width, label=reg.capitalize(),
                   color=colors[reg], alpha=0.85)

    short_prompt = prompt[:40] + "..."
    ax.set_title(short_prompt, fontsize=9)
    ax.set_xticks(x + 1.5*width)
    ax.set_xticklabels([f"F{FEATURES_REGISTRO[fn]}" for fn in feat_names],
                       rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Ativação média")
    ax.legend(fontsize=7)

plt.suptitle("Ativação de Features por Registro Amplificado", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("fig_phase6_register_steering.png", dpi=150, bbox_inches="tight")
if os.path.exists(SAVE_DIR):
    plt.savefig(SAVE_DIR + "fig_phase6_register_steering.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: fig_phase6_register_steering.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sweep_acts_mean = []
sweep_acts_max = []

for r in results_sweep:
    acts = measure_feature_activations(model, sae, tokenizer, r["text"], [2294])
    sweep_acts_mean.append(acts[2294]["mean"])
    sweep_acts_max.append(acts[2294]["max"])

ax.plot(sweep_multipliers, sweep_acts_mean, 'o-', label="Ativação média (F2294)",
        color="#8B0000", linewidth=2, markersize=8)
ax.plot(sweep_multipliers, sweep_acts_max, 's--', label="Ativação máxima (F2294)",
        color="#B71C1C", linewidth=1.5, markersize=6, alpha=0.7)

ax.set_xlabel("Multiplicador de Steering")
ax.set_ylabel("Ativação da Feature 2294")
ax.set_title("Sweep de Multiplicadores — Feature Jurídica (2294)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fig_phase6_multiplier_sweep.png", dpi=150, bbox_inches="tight")
if os.path.exists(SAVE_DIR):
    plt.savefig(SAVE_DIR + "fig_phase6_multiplier_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: fig_phase6_multiplier_sweep.png")

## 7. Summary and Saving of Results

In [ ]:
all_results = {
    "ablation": {
        "crase": results_crase,
        "genero": results_genero,
        "cliticos": results_cliticos,
    },
    "register_steering": results_registro,
    "multiplier_sweep": results_sweep,
    "config": {
        "features_pt": FEATURES_PT,
        "features_registro": FEATURES_REGISTRO,
        "register_multiplier": REGISTER_MULT,
        "sweep_multipliers": sweep_multipliers,
        "model": "gemma-2-2b",
        "sae": SAE_ID,
        "layer": LAYER,
    }
}

save_path = "phase6_steering_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print(f"Results saved: {save_path}")

if os.path.exists(SAVE_DIR):
    drive_path = SAVE_DIR + "phase6_steering_results.json"
    with open(drive_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"Results saved to Drive: {drive_path}")

print()
print("=" * 70)
print("PHASE 6 SUMMARY")
print("=" * 70)
print(f"Ablation experiments: 3 (crase, gender, clitics)")
print(f"  Crase prompts: {len(results_crase)}")
print(f"  Gender prompts: {len(results_genero)}")
print(f"  Clitics prompts: {len(results_cliticos)}")
print(f"Register experiments: 4 registers × {len(prompts_register)} prompts")
print(f"Sweep de multiplicadores: {len(sweep_multipliers)} valores")
print(f"Figuras geradas: 3")
print(f"  - fig_phase6_ablation_comparison.png")
print(f"  - fig_phase6_register_steering.png")
print(f"  - fig_phase6_multiplier_sweep.png")
print()

In [ ]:
# placeholder